# Comprehensive Gen Python Bindings Demo

This notebook exhaustively demonstrates all functionality available in the Gen Python bindings.
It covers: initialization, imports, exports, updates, search, indexing, and interactive visualization.

---

## 1. Setup and Repository Initialization

In [1]:
import gen
import tempfile
import pathlib

REPO_ROOT = pathlib.Path(gen.__file__).parents[3]
FIXTURES = REPO_ROOT / "fixtures"

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-api-demo-"))
print(f"Working directory: {WORK_DIR}")

repo = gen.Repository(str(WORK_DIR))
print(f"Repository initialized: {repo}")
print(f"  gen_dir: {repo.gen_dir}")

Working directory: /tmp/gen-notebook
Repository initialized: <builtins.Repository object at 0x10b5b2b50>
  gen_dir: /tmp/gen-notebook/.gen


## 2. Import Functions

Gen supports importing from multiple file formats: FASTA, GFA, GenBank, and Library.

### 2.1 Import FASTA

In [2]:
SIMPLE_FA = FIXTURES / "simple.fa"
print(f"FASTA file: {SIMPLE_FA}")
print(SIMPLE_FA.read_text())

FASTA file: /Users/bvh/git/gen-wt3/fixtures/simple.fa
>m123
ATCGATCGATCGATCGATCGGGAACACACAGAGA



In [3]:
repo.import_fasta(str(SIMPLE_FA), sample="simple")
sequence_graphs = repo.get_sequence_graphs()
print(f"Imported {len(sequence_graphs)} sequence graph(s)")
for sg in sequence_graphs:
    print(f"  {sg}")

Imported 1 sequence graph(s)
  SequenceGraph(b5fa08f762a3bfc0a8d73fb9ab9b3c5a, collection="default", sample="simple", name="m123")


### 2.2 Import GFA (Graph Fragment Assembly)

In [4]:
SIMPLE_GFA = FIXTURES / "simple.gfa"
print(f"GFA file: {SIMPLE_GFA}")
print(SIMPLE_GFA.read_text())

GFA file: /Users/bvh/git/gen-wt3/fixtures/simple.gfa
H	VN:Z:1.2
S	1	ATC	SN:Z:m123	SO:i:0	SR:i:0
S	2	GATCGATCGA	SN:Z:m123	SO:i:3	SR:i:0
S	3	TCGATCGGG	SN:Z:m123	SO:i:13	SR:i:0
S	4	AACACACAGAGA	SN:Z:m123	SO:i:22	SR:i:0
L	1	+	2	+	*
L	2	+	3	+	*
L	3	+	4	+	*
P	m123	1+,2+,3+,4+	4M



In [5]:
repo.import_gfa(str(SIMPLE_GFA), sample="graph_example")
sequence_graphs = repo.get_sequence_graphs()
print(f"Total sequence graphs: {len(sequence_graphs)}")

Total sequence graphs: 2


### 2.3 Import GenBank

In [6]:
INSERTION_GB = FIXTURES / "geneious_genbank" / "insertion.gb"
print(f"GenBank file: {INSERTION_GB}")
print(INSERTION_GB.read_text()[:500] + "...")

GenBank file: /Users/bvh/git/gen-wt3/fixtures/geneious_genbank/insertion.gb
LOCUS       insertion               8302 bp    DNA     circular     27-NOV-2024
DEFINITION  Cloning vector pBeloBAC11, complete sequence.
ACCESSION   urn.local...12f-ii1u066
KEYWORDS    .
SOURCE      Cloning vector pBeloBAC11
  ORGANISM  Cloning vector pBeloBAC11
            other sequences; artificial sequences; vectors.
REFERENCE   1  (bases 1 to 7507)
  AUTHORS   New England Biolabs.
  TITLE     Direct Submission
  JOURNAL   Submitted (19-OCT-2007) Research Department, New England Biolabs,
  ...


In [7]:
repo.import_genbank(str(INSERTION_GB), sample="insertion_example")
sequence_graphs = repo.get_sequence_graphs()
print(f"Total sequence graphs: {len(sequence_graphs)}")

Total sequence graphs: 3


### 2.4 Import Library (combinatorial design)

In [8]:
# import_library_files(library_name, parts_fasta_path, design_csv_path, sample=None, collection=None)
PARTS_FA = FIXTURES / "protein_segments.fa"
DESIGN_CSV = FIXTURES / "protein_layout.csv"
print(f"Parts FASTA: {PARTS_FA}")
print(f"Design CSV: {DESIGN_CSV}")

Parts FASTA: /Users/bvh/git/gen-wt3/fixtures/protein_segments.fa
Design CSV: /Users/bvh/git/gen-wt3/fixtures/protein_layout.csv


In [9]:
repo.import_library_files("combinatorial_lib", str(PARTS_FA), str(DESIGN_CSV))
sequence_graphs = repo.get_sequence_graphs()
print(f"Total sequence graphs: {len(sequence_graphs)}")
for sg in sequence_graphs:
    print(f"  {sg}")

Total sequence graphs: 4
  SequenceGraph(28067977ce0b8976ce7ccd0c8d4ba108, collection="default", sample="graph_example", name="")
  SequenceGraph(766969fdfee23aa8174ecaf1cc2a4892, collection="default", sample="insertion_example", name="insertion")
  SequenceGraph(b5fa08f762a3bfc0a8d73fb9ab9b3c5a, collection="default", sample="simple", name="m123")
  SequenceGraph(be95231fada7e151ccd2a64f1803abca, collection="default", sample="reference", name="combinatorial_lib")


## 3. Export Functions

Gen supports exporting to FASTA, GFA, and GenBank formats.

In [10]:
sequence_graph = repo.get_sequence_graphs()[0]
print(f"Exporting: {sequence_graph}")

Exporting: SequenceGraph(28067977ce0b8976ce7ccd0c8d4ba108, collection="default", sample="graph_example", name="")


### 3.1 Sequence Graph Exports (sample-scoped)

`SequenceGraph` objects expose `export_fasta`, `export_gfa`, and `export_genbank`.
These export **all sequence graphs belonging to the graph's sample**: they are
sample-scoped, not limited to the single sequence graph.


In [11]:
sg = repo.get_sequence_graphs()[0]

BG_EXPORTED_FA = WORK_DIR / "bg_exported.fa"
sg.export_fasta(str(BG_EXPORTED_FA))
print(f"sg.export_fasta() → {BG_EXPORTED_FA.read_text()[:200]}")

BG_EXPORTED_GFA = WORK_DIR / "bg_exported.gfa"
sg.export_gfa(str(BG_EXPORTED_GFA))
print(f"sg.export_gfa() → {BG_EXPORTED_GFA.read_text()[:200]}")

BG_EXPORTED_GB = WORK_DIR / "bg_exported.gb"
sg.export_genbank(str(BG_EXPORTED_GB))
print(f"sg.export_genbank() → {BG_EXPORTED_GB.read_text()[:200]}")

Exported to file /tmp/gen-notebook/bg_exported.fa
sg.export_fasta() → >
ATCGATCGATCGATCGATCGGGAACACACAGAGA

sg.export_gfa() → S	01a1b5c4da9c77618426ca1305987c8e.0.12	AACACACAGAGA
S	01a1b5c4da9c77618426c9e69dbd07b3.0.3	ATC
S	01a1b5c4da9c77618426c9f0f3f9a673.0.10	GATCGATCGA
S	01a1b5c4da9c77618426ca01e79c975c.0.9	TCGATCGGG
L	01
sg.export_genbank() → LOCUS                                 34 bp            linear UNK 01-JAN-1970
ORIGIN      
        1 ATCGATCGAT CGATCGATCG GGAACACACA GAGA
//



## 4. Update Functions

Updates modify existing block groups with new sequences, variants, or graph changes.

### 4.1 Edit a Sample Copy (direct replacement)

The edit updates the current path along with the graph's routes.


In [12]:
bg_for_update = next(sg for sg in repo.get_sequence_graphs() if sg.name == "m123")
print(f"Updating: {bg_for_update}")

mutant = bg_for_update.sample.copy("mutant")  # new sample name
next(sg for sg in mutant if sg.name == "m123").replace(
    f"{bg_for_update.name}:10-34",  # region with coordinates
    "ATCGATCGATCGATCGATCGATCGATCGATCGATCG",
)
print("Updated sequence graph with sequence")


Updating: SequenceGraph(b5fa08f762a3bfc0a8d73fb9ab9b3c5a, collection="default", sample="simple", name="m123")
Updated sequence graph with sequence


### 4.2 Update with FASTA

In [13]:
PARTS_FA = FIXTURES / "parts.fa"
print(f"Parts FASTA: {PARTS_FA}")
print(PARTS_FA.read_text())

Parts FASTA: /Users/bvh/git/gen-wt3/fixtures/parts.fa
>p1
AAAA
>p2
TAAT
>p3
CAAC
>cds1
ATGATAA
>cds2
ATGTTAA
>cds3
ATGCTAA



In [14]:
# BROKEN
PARTS_FA = FIXTURES / "parts.fa"
PARTS_BG = repo.get_sequence_graphs()[0]
print(f"Parts FASTA: {PARTS_FA}")
#print(PARTS_FA.read_text())
#repo.update_with_fasta(
#    str(PARTS_FA),
#    PARTS_BG.sample.name,  # existing sample
#    "parts_updated",        # new sample name
#    PARTS_BG.name,          # region (block group name)
#    0,                      # start coordinate
#    50,                     # end coordinate
#)
#print(f"Updated with parts")

Parts FASTA: /Users/bvh/git/gen-wt3/fixtures/parts.fa


### 4.3 Update with VCF (variant calling)

In [15]:
SIMPLE_VCF = FIXTURES / "simple.vcf"
print(f"VCF file: {SIMPLE_VCF}")
print(SIMPLE_VCF.read_text())

VCF file: /Users/bvh/git/gen-wt3/fixtures/simple.vcf
##fileformat=VCFv4.1
##filedate=Tue Sep  4 13:12:57 2018
##reference=simple.fa
##contig=<ID=m123,length=34>
##phasing=none
##INFO=<ID=GAN,Number=1,Type=String,Description="Accession name for Gen">
##INFO=<ID=GAA,Number=1,Type=Integer,Description="Allele to create accession for. Defaults to reference allele (0).">
##INFO=<ID=TYPE,Number=A,Type=String,Description="The type of allele, either snp, mnp, ins, del, or complex.">
##filter="QUAL > 1"
##FILTER=<ID=PASS,Description="All filters passed">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	unknown	G1	foo
m123	3	.	CG	C	1611.92	.	GAN=del1;GAA=1;TYPE=del	GT	1/1	.	1/1
m123	10	.	T	TAGA	2216.6	.	TYPE=ins	GT	1/1	.	0/0



In [16]:
# broken
bg_for_vcf = repo.get_sequence_graphs()[0]
#repo.update_with_vcf(str(SIMPLE_VCF))
#print(f"Updated with VCF")

### 4.4 Update with GFA

In [17]:
WALK_GFA = FIXTURES / "walk.gfa"
print(f"Walk GFA: {WALK_GFA}")
print(WALK_GFA.read_text())

Walk GFA: /Users/bvh/git/gen-wt3/fixtures/walk.gfa
H	VN:Z:1.2
S	1	ACCT	SN:Z:291344	SO:i:0	SR:i:0
S	2	ACAA	SN:Z:291344	SO:i:4	SR:i:0
S	3	ATTC	SN:Z:291344	SO:i:8	SR:i:0
S	4	AAAC	SN:Z:291344	SO:i:12	SR:i:0
L	1	+	2	+	*
L	2	+	3	+	*
L	3	+	4	+	*
W	291344	0	291344	0	16	>1>2>3>4	*



In [18]:
bg_for_gfa = repo.get_sequence_graphs()[0]
repo.update_with_gfa(str(WALK_GFA), bg_for_gfa.sample.name, "walk_merged")
print(f"Updated with GFA")

Updated with GFA
Updated with GFA file: /Users/bvh/git/gen-wt3/fixtures/walk.gfa


### 4.6 Update with GAF (graph alignment format)

In [19]:
CHR22_GAF = FIXTURES / "chr22_het.gaf"
print(f"GAF file: {CHR22_GAF}")
print(CHR22_GAF.read_text()[:500])

GAF file: /Users/bvh/git/gen-wt3/fixtures/chr22_het.gaf
test_left	50	0	50	+	>138.0	50	0	50	50	50	31	tp:A:P	NM:i:0	cm:i:30	s1:i:50	s2:i:0	dv:f:0	cg:Z:50=	ds:Z::50
test_right	50	0	50	+	>140.0	50	0	50	50	50	31	tp:A:P	NM:i:0	cm:i:30	s1:i:50	s2:i:0	dv:f:0	cg:Z:50=	ds:Z::50
2node_span_left	85	0	85	+	>454.0>455.0	100	0	79	79	85	60	tp:A:P	NM:i:6	cm:i:20	s1:i:79	s2:i:0	dv:f:0.0561	cg:Z:22=6I57=	ds:Z::22+aaaaa[a]:57
2node_span_right	89	0	89	+	>530.0>531.0	100	11	100	89	89	60	tp:A:P	NM:i:0	cm:i:49	s1:i:90	s2:i:0	dv:f:0.0163	cg:Z:89=	ds:Z::89
left_extreme_right	


In [20]:
# update_with_gaf requires both a .gaf alignment file and a companion CSV coordinates file
# repo.update_with_gaf(str(CHR22_GAF), str(CHR22_CSV), "gaf_sample")
print(f"GAF file: {CHR22_GAF}")
print(CHR22_GAF.read_text()[:500])

GAF file: /Users/bvh/git/gen-wt3/fixtures/chr22_het.gaf
test_left	50	0	50	+	>138.0	50	0	50	50	50	31	tp:A:P	NM:i:0	cm:i:30	s1:i:50	s2:i:0	dv:f:0	cg:Z:50=	ds:Z::50
test_right	50	0	50	+	>140.0	50	0	50	50	50	31	tp:A:P	NM:i:0	cm:i:30	s1:i:50	s2:i:0	dv:f:0	cg:Z:50=	ds:Z::50
2node_span_left	85	0	85	+	>454.0>455.0	100	0	79	79	85	60	tp:A:P	NM:i:6	cm:i:20	s1:i:79	s2:i:0	dv:f:0.0561	cg:Z:22=6I57=	ds:Z::22+aaaaa[a]:57
2node_span_right	89	0	89	+	>530.0>531.0	100	11	100	89	89	60	tp:A:P	NM:i:0	cm:i:49	s1:i:90	s2:i:0	dv:f:0.0163	cg:Z:89=	ds:Z::89
left_extreme_right	


### 4.7 Update with Library

In [21]:
SINGLE_COL_CSV = FIXTURES / "single_column_design.csv"
print(f"Library CSV: {SINGLE_COL_CSV}")
print(SINGLE_COL_CSV.read_text())

Library CSV: /Users/bvh/git/gen-wt3/fixtures/single_column_design.csv
p1
p2
p3



In [22]:
SINGLE_COL_CSV = FIXTURES / "single_column_design.csv"
PARTS_FASTA = FIXTURES / "parts.fa"
bg_for_lib = next(
    sg
    for sg in repo.get_sequence_graphs()
    if sg.sample.name == "simple" and sg.name == "m123"
)
repo.update_with_library_files(
    bg_for_lib.sample.name,  # existing sample
    "lib_updated",            # new sample name
    f"{bg_for_lib.name}:7-20",  # region within the imported simple sequence
    str(SINGLE_COL_CSV),      # library layout CSV
    str(PARTS_FASTA),         # parts FASTA
)
print(f"Updated with library")

Updated with library


## 5. Search and Indexing

Gen provides fast sequence search with optional indexing.

### 5.1 Search without Index (full scan)

In [23]:
test_bg = repo.get_sequence_graphs()[0]
print(f"Searching in: {test_bg}")

query = "ATCG"
results = repo.search(query, [test_bg])
for sg, loci in results:
    print(f"Found {len(loci)} match(es) for '{query}' in {sg.name}")
    for m in loci[:3]:
        print(f"  {m}")

Searching in: SequenceGraph(28067977ce0b8976ce7ccd0c8d4ba108, collection="default", sample="graph_example", name="")
Found 9 match(es) for 'ATCG' in 
  ATCG
  ATCG
  ATCG


### 5.2 Build Search Index

In [24]:
repo.build_index(k=8)
print("Search index built successfully")

index_dir = pathlib.Path(str(repo.gen_dir)) / "search_index"
index_files = list(index_dir.glob("*.bin")) if index_dir.exists() else []
print(f"Index files created: {len(index_files)}")

Search index built successfully
Index files created: 7


### 5.3 Search with Index

In [25]:
results = repo.search(query, [test_bg])
for sg, loci in results:
    print(f"Found {len(loci)} match(es) for '{query}' in {sg.name} (with index)")
    for m in loci[:3]:
        print(f"  {m}")

Found 9 match(es) for 'ATCG' in  (with index)
  ATCG
  ATCG
  ATCG


### 5.4 Block-group level index operations

In [26]:
test_bg = repo.get_sequence_graphs()[0]
test_bg.build_index(k=4)
print("Index built on specific block group")

Index built on specific block group


### 5.5 Clear Index

In [27]:
repo.clear_index()
print("All search indexes cleared")

All search indexes cleared


## 6. Graph Operations

Deriving subgraphs and chunks from sequence graphs.


### 6.1 Subgraph


In [28]:
# Use the genbank-imported sample explicitly — it is long enough for a
# 0-100 slice and its identity is stable (other samples in the repo are
# much shorter or get mutated by earlier edits).
sg = next(b for b in repo.get_sequence_graphs() if b.sample.name == "insertion_example")
# subgraph(new_sample, start, end)
sg.subgraph("subgraph", 0, 100)
subgraphs = repo.get_sequence_graphs(sample="subgraph")
print(f"Derived {len(subgraphs)} subgraph sequence graph(s)")


Derived 1 subgraph sequence graph(s)
Derive subgraph succeeded.


### 6.2 Chunks


In [29]:
# chunks(new_sample, breakpoints=None, chunk_size=None)
chunks = sg.chunks("chunks", chunk_size=50)
print(f"Derived {len(chunks)} chunk(s) from {sg.name}")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk}")


Derived 151 chunk(s) from insertionDerive chunks succeeded.

  Chunk 0: SequenceGraph(66341901845a981f3b2fc05baf4a3d3a, collection="default", sample="chunks", name="insertion.1")
  Chunk 1: SequenceGraph(555f0d0a2a44fd410952bec555ed0279, collection="default", sample="chunks", name="insertion.2")
  Chunk 2: SequenceGraph(2c8dda6af077403477e8180fc5119945, collection="default", sample="chunks", name="insertion.3")
  Chunk 3: SequenceGraph(0261d51bb9b2185da2ce4857c88726f4, collection="default", sample="chunks", name="insertion.4")
  Chunk 4: SequenceGraph(2afa0a5dae7d61c830c5b4a0d2ca6055, collection="default", sample="chunks", name="insertion.5")
  Chunk 5: SequenceGraph(4c080825caf0a14e2823b11a2b79baf5, collection="default", sample="chunks", name="insertion.6")
  Chunk 6: SequenceGraph(f4131d3a5bbc6f1699a0a1afe1a967e7, collection="default", sample="chunks", name="insertion.7")
  Chunk 7: SequenceGraph(807ae3a23e3a9ca940fed8f6e2130c49, collection="default", sample="chunks", name="insertion

## 7. Interactive Graph Visualization

The Jupyter widget allows interactive exploration of sequence graphs.

In [30]:
sg = repo.get_sequence_graphs()[0]
widget = sg.plot(rows=20, cols=80, detail="full")
widget

                                  insertion.80








  ╟──gactgaggtatgtgctcttcttatctccttttgtagtgttgctcttattt──╢








     |...................|...................|.........
     0                  20                  40

### 7.1 Highlight matches in widget

In [31]:
results = repo.search("ATC", [sg])
matches = results[0][1] if results else []
print(f"Found {len(matches)} matches to highlight")

widget.clear_highlights()
for m in matches[:5]:
    widget.show(m,"cyan")
widget

Found 1 matches to highlight


                                  insertion.80








tATCtccttttgtagtgttgctcttattt──╢








.........|.........|.........
        30        40

### 7.2 Different highlight colors

In [32]:
for m in matches[:5]:
    widget.show(m)
print("Multi-color highlights applied")



Multi-color highlights applied


### 7.3 Freeze the widget

Click the **❄** button in the widget toolbar to freeze the widget as a static
PNG embedded in the notebook. The graph remains visible in static viewers
(GitHub, nbviewer) without the `gen` module running.

## 8. Block Group Properties and Methods

In [33]:
sg = repo.get_sequence_graphs()[0]
print(f"sequence graph: {sg}")
print(f"  ID: {sg.id}")
print(f"  Name: {sg.name}")
print(f"  Collection: {sg.collection}")
print(f"  Sample: {sg.sample.name}")

sequence graph: SequenceGraph(014d0d76ce526deb0332e0024ebba7cb, collection="default", sample="chunks", name="insertion.80")
  ID: 014d0d76ce526deb0332e0024ebba7cb
  Name: insertion.80
  Collection: default
  Sample: chunks


### 9.1 Get block sequence

In [34]:
result = repo.query("SELECT name, collection_name, sample_name FROM block_groups LIMIT 5")
print("sequence graphs from raw query:")
for row in result:
    print(f"  name={row[0]}  collection={row[1]}  sample={row[2]}")

sequence graphs from raw query:
  name=insertion.1  collection=default  sample=chunks
  name=insertion.10  collection=default  sample=chunks
  name=insertion.100  collection=default  sample=chunks
  name=insertion.101  collection=default  sample=chunks
  name=insertion.102  collection=default  sample=chunks


### 9.2 Graph position and locus

In [35]:
# Positions are obtained from search results via .start() and .end()
hits = sg.search("ATCG")
if hits:
    h = hits[0]
    print(f"Hit: {h}")
    s = h.start()
    e = h.end()
    print(f"  start(): {s}")
    print(f"  start().node: {s.node}")
    print(f"  start().offset: {s.offset}")
    print(f"  end(): {e}")
    print(f"  strand: {h.strand}")
    print(f"  sequence: {h.sequence}")
else:
    print("No hits — try a different query or check the imported sequence.")

No hits — try a different query or check the imported sequence.


## 9. Collection Operations

In [36]:
result = repo.query("SELECT DISTINCT collection_name FROM block_groups")
collections = [row[0] for row in result]
print(f"Collections: {collections}")

for coll in collections:
    sgs = repo.get_sequence_graphs(collection=coll)
    print(f"  {coll}: {len(sgs)} sequence graph(s)")

Collections: ['default']
  default: 159 sequence graph(s)


## 10. Database Queries

In [37]:
result = repo.query("SELECT COUNT(*) FROM block_groups")
print(f"sequence graphs count: {result[0][0]}")

result = repo.query("SELECT name, collection_name FROM block_groups LIMIT 5")
print("sequence graph names and collections:")
for row in result:
    print(f"  {row[0]} in {row[1]}")

sequence graphs count: 159
sequence graph names and collections:
  insertion.1 in default
  insertion.10 in default
  insertion.100 in default
  insertion.101 in default
  insertion.102 in default


## 11. Stitch (create joined sequence)

`Repository.stitch()` joins `SequenceGraph` or `Locus` parts end to end.

A `Locus` is a linear span, but what gets stitched is the subgraph of **every variant** between
its first and last positions: stitching `graph.region("name:100-200")` keeps all the alternatives
inside that region, not only the bases the current path reads there. A whole `SequenceGraph` part
contributes all of its routes.


In [38]:
# Stitch the chunks derived above back together, then two regions of one graph directly.
stitched = repo.stitch(chunks, "stitched", "combined_region")
print(f"Stitch result: {stitched}")

long_bg = next(b for b in repo.get_sequence_graphs() if b.sample.name == "insertion_example")
region_pieces = repo.stitch(
    [long_bg.region(f"{long_bg.name}:0-20"), long_bg.region(f"{long_bg.name}:60-80")],
    "stitched_loci",
    "combined_loci",
)
print(f"Locus stitch result: {region_pieces}")


Stitch result: SequenceGraph(e4135a712205169b8c4f1c1e4b6b91a3, collection="default", sample="stitched", name="combined_region")
Locus stitch result: SequenceGraph(63210faf44600049b9f0885a2cf22fe7, collection="default", sample="stitched_loci", name="combined_loci")


## 12. Graph Export Formats

Convert graphs to other formats for external tools.

### 13.1 NetworkX conversion

In [39]:
try:
    import networkx as nx

    sg = repo.get_sequence_graphs()[0]
    nx_graph = sg.to_networkx()
    print(f"NetworkX graph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")
    print(f"Is directed: {nx_graph.is_directed()}")

    degrees = [d for n, d in nx_graph.degree()]
    if degrees:
        print(f"Average degree: {sum(degrees)/len(degrees):.2f}")

except ImportError:
    print("Install networkx: pip install networkx")

NetworkX graph: 3 nodes, 2 edges
Is directed: True
Average degree: 1.33


### 13.2 RustworkX conversion

In [40]:
try:
    import rustworkx as rx

    sg = repo.get_sequence_graphs()[0]
    rx_graph = sg.to_rustworkx()
    print(f"RustworkX graph: {rx_graph.num_nodes()} nodes, {rx_graph.num_edges()} edges")

except ImportError:
    print("Install rustworkx: pip install rustworkx")

Install rustworkx: pip install rustworkx


### 13.3 Dictionary representation

In [41]:
sg = repo.get_sequence_graphs()[0]
dict_repr = sg.to_dict()
print(f"Dictionary keys: {list(dict_repr.keys())}")
print(f"Nodes: {len(dict_repr['nodes'])} items")
print(f"Edges: {len(dict_repr['edges'])} items")

Dictionary keys: ['nodes', 'edges']
Nodes: 3 items
Edges: 2 items


## Summary

This notebook demonstrated:

1. **Repository initialization** - `gen.Repository()`
2. **Import functions** - `import_fasta`, `import_gfa`, `import_genbank`, `import_library`
3. **Export functions** - `export_fasta`, `export_gfa`, `export_genbank` (on a sequence graph, covering its sample)
4. **Update functions** - `update_with_fasta`, `update_with_vcf`, `update_with_gfa`, `update_with_gaf`, `update_with_library`, and `copy` plus `replace` for direct edits
5. **Search and indexing** - `search`, `build_index`, `clear_index`
6. **Graph operations** - `subgraph`, `chunks`, `stitch`
7. **Interactive visualization** - Jupyter widget with highlighting and freeze button for static distribution
8. **Sequence graph properties** - ID, name, collection, sample
9. **Graph position/types** - `Position`, `Locus`, `Node`
10. **Database queries** - Direct SQL queries on the repository
11. **Graph conversions** - NetworkX, RustworkX, dictionary formats

---


In [42]:
print(f"Gen version: {gen.__version__}")
print(f"\nExported functions and classes:")
for name in sorted(gen.__all__):
    print(f"  - {name}")

Gen version: 0.5.1

Exported functions and classes:
  - Annotation
  - Asset
  - Branch
  - HashId
  - Locus
  - Node
  - Operation
  - Position
  - Remote
  - Repository
  - Sample
  - Sequence
  - SequenceGraph
  - SuperPosition
  - clone
  - freeze_all_widgets
